In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "test_notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from omegaconf import OmegaConf

from src.ml.build import (
    build_birefnet,
    build_dl,
    build_lora_birefnet_for_training,
)
from src.ml.training.loss import CustomLoss

In [2]:
cfg = OmegaConf.merge(
    OmegaConf.load("src/config/tune.yaml"),
    OmegaConf.load("src/config/model.yaml"),
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

device: cuda


In [3]:
train_dl, valid_dl, split_filenames = build_dl(cfg)
print(f"train batches: {len(train_dl)}  |  valid batches: {len(valid_dl)}")

train batches: 209  |  valid batches: 23


In [4]:
base_model = build_birefnet(cfg=cfg).to(device)
model = build_lora_birefnet_for_training(cfg=cfg, model=base_model)
print(
    f"total={model.stats['total']:,}  trainable={model.stats['trainable']:,}  "
    f"ratio={model.stats['trainable'] / model.stats['total']:.2%}"
)

[LOAD] weight/BiRefNet-general-epoch_244.pth


total=223,728,282  trainable=3,551,784  ratio=1.59%


In [5]:
criterion = CustomLoss(
    lambda_bce=cfg.loss.lambda_bce,
    lambda_iou=cfg.loss.lambda_iou,
    lambda_kl=cfg.loss.lambda_kl,
    lambda_aux=cfg.loss.lambda_aux,
)

In [6]:
batch = next(iter(train_dl))
print(
    f"image_1: {tuple(batch['image_1'].shape)}  "
    f"image_2: {tuple(batch['image_2'].shape)}  "
    f"mask: {tuple(batch['mask'].shape)}"
)

image_1: (1, 3, 1024, 1024)  image_2: (1, 3, 1024, 1024)  mask: (1, 1, 1024, 1024)


In [7]:
model.train()
loss_dict, loss = criterion(model, batch)

print("[train loss]")
for k, v in loss_dict.items():
    print(f"  {k:>5}: {v.item():.4f}")

[train loss]
   loss: 11.9417
    seg: 11.2480
   cons: 0.0006
    aux: 0.6931


In [8]:
loss.backward()

grad_params = [p for p in model.get_adapter_params() if p.grad is not None]
total_norm = torch.norm(
    torch.stack([p.grad.detach().norm(2) for p in grad_params])
).item()
print(f"adapter params with grad: {len(grad_params)}")
print(f"grad L2 norm: {total_norm:.4f}")

adapter params with grad: 282
grad L2 norm: 9.6924
